In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
from catboost import CatBoostRanker

K = 50
MODEL_FILE = 'ranker_enc.cbm'          # итоговая модель CatBoost (с признаками энкодера)
MC_FEATURES = ['mc_proba', 'mc_proba_rel', 'mc_top1_proba']
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

def find_all(name):
    return [p for root in ['/kaggle/input', '.'] for p in sorted(glob.glob(os.path.join(root, '**', name), recursive=True))]

FEAT_DIR = os.path.dirname(find_all('features_v2.json')[0])
MC_PATH = find_all('microcat_proba.parquet')[0]
MODEL_PATH = find_all(MODEL_FILE)[0]
SUBMITTED = [p for p in find_all('answer.csv') if os.path.abspath(p) != os.path.abspath(f'{OUT}/answer.csv')]
print('признаки:', FEAT_DIR, '\nклассификатор:', MC_PATH, '\nмодель:', MODEL_PATH, '\nотправленный ответ:', SUBMITTED[:1])

### Признаки бенчмарка + признаки классификатора (так же, как при обучении)

In [ ]:
bench = pd.read_parquet(f'{FEAT_DIR}/bench.parquet')

mc = pd.read_parquet(MC_PATH)
mc_top1 = mc.groupby('qid')['proba'].max().rename('mc_top1_proba')
mc = mc.rename(columns={'microcat': 'item_microcat_id', 'proba': 'mc_proba'})
bench = bench.merge(mc[['qid', 'item_microcat_id', 'mc_proba']], on=['qid', 'item_microcat_id'], how='left')
bench = bench.merge(mc_top1, left_on='qid', right_index=True, how='left')
bench['mc_proba'] = bench['mc_proba'].fillna(0).astype('float32')
bench['mc_proba_rel'] = (bench['mc_proba'] / bench['mc_top1_proba'].clip(lower=1e-6)).astype('float32')
print(f'предсказания классификатора есть у {bench.qid.drop_duplicates().isin(set(mc.qid)).mean():.2%} запросов бенчмарка')

bench['item_microcat_id'] = bench['item_microcat_id'].astype('int64').astype(str)   # категориальный, как при обучении
bench = bench.sort_values('qid', kind='stable').reset_index(drop=True)

model = CatBoostRanker().load_model(MODEL_PATH)
bench['score'] = model.predict(bench[model.feature_names_])        # список признаков берём из самой модели
print(f'бенчмарк: {bench.qid.nunique():,} запросов, {len(bench):,} кандидатов; признаков у модели: {len(model.feature_names_)}')

In [ ]:
def build_answer(df):
    b = df[['qid', 'item_id', 'score']].sort_values(['qid', 'score'], ascending=[True, False]).drop_duplicates(['qid', 'item_id'])
    top = b.groupby('qid').head(K)
    return (top.groupby('qid', sort=False)['item_id'].apply(' '.join).reset_index()
            .rename(columns={'qid': 'query_id', 'item_id': 'answer'}))

answer = build_answer(bench)
if SUBMITTED:
    sub = pd.read_csv(SUBMITTED[0], dtype=str, keep_default_na=False).set_index('query_id')['answer']
    same = (answer.set_index('query_id')['answer'].reindex(sub.index) == sub).mean()
    print(f'совпадает с отправленным ответом в {same:.2%} строк')

In [ ]:
assert answer.columns.tolist() == ['query_id', 'answer']
assert answer.query_id.is_unique and answer.query_id.str.len().eq(16).all()
ids = answer.answer.str.split(' ')
assert ids.map(len).between(1, K).all() and ids.map(lambda x: len(set(x)) == len(x)).all()
assert ids.explode().str.fullmatch(r'[0-9a-f]{16}').all()
answer.to_csv(f'{OUT}/answer.csv', index=False)
print(f'{OUT}/answer.csv: {len(answer):,} строк (ожидается 2452)')